In [1]:
#run spark.stop before running this notebook. some of the configs won't take if an active spark sessions is grabbed or running. 
#spark.stop()

In [2]:
import psutil
import time
import threading
import datetime
import json
import xgboost as xgb

In [3]:
import pyspark
import os
import pandas as pd
from torchvision.io import read_image
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import datasets, models
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import pyarrow
from io import BytesIO
import base64



from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import FloatType

import math
from statistics import NormalDist

In [4]:
class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = torch.nn.Sequential(
            #Input = 3 x 32 x 32, Output = 32 x 32 x 32 
            #new input 3x600x375 output=
            torch.nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 32 x 32 x 32, Output = 32 x 16 x 16 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 32 x 16 x 16, Output = 64 x 16 x 16 
            torch.nn.Conv2d(in_channels = 6, out_channels = 12, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 16 x 16, Output = 64 x 8 x 8 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8 
            torch.nn.Conv2d(in_channels = 12, out_channels = 24, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8 
            torch.nn.Conv2d(in_channels = 24, out_channels = 48, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8 
            torch.nn.Conv2d(in_channels = 48, out_channels = 96, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4 
            torch.nn.MaxPool2d(kernel_size=2),

            torch.nn.Flatten(),

            torch.nn.Linear(96*11*18, 128),
            torch.nn.ReLU(),
            torch.nn.BatchNorm1d(128),




            #torch.nn.Linear(128, 128),
            #torch.nn.ReLU(),
            #torch.nn.BatchNorm1d(128),



            torch.nn.Linear(128, 2),
            torch.nn.ReLU(),
            #torch.nn.BatchNorm1d(2),
            #torch.nn.Dropout(0.2),


            torch.nn.Softmax(dim=1)

        )

    def forward(self, x):

            

        return self.model(x)

In [5]:
#tuning knob 1. put a very between 1 and 10, depending on sys cores. this also reparitions the df for sql queries
threads = 4

In [6]:
spark = pyspark.sql.SparkSession.builder.master(f"local[{threads}]").appName("silver").config("spark.executor.memory", "24g").config("spark.driver.memory", "24g").getOrCreate()

print(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/05 23:40:45 WARN Utils: Your hostname, research-ub22, resolves to a loopback address: 127.0.1.1; using 192.168.0.54 instead (on interface ens18)
25/10/05 23:40:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/05 23:40:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNN().to(device)
model.load_state_dict(torch.load('../mymodel', map_location=torch.device(device)))
model.eval() # Set the model to evaluation mode

bc_model = spark.sparkContext.broadcast(model)

In [8]:
#second_model
res18_model = models.resnet18(pretrained=True).to(device)
res18_model.fc = nn.Linear(res18_model.fc.in_features, 2 )
res18_model.load_state_dict(torch.load('../resnet18_model.pth', map_location=torch.device(device)))
res18_model.eval() # Set the model to evaluation mode

bc_res18_model = spark.sparkContext.broadcast(res18_model)

/home/starfuryz2040/DataspellProjects/research/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/starfuryz2040/DataspellProjects/research/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [9]:
print(device)

cpu


NORMAL UDF (doesnt work in spark sql)

In [10]:
def predict_tensor(base64image):
    img_data = base64.b64decode(base64image)
    image = Image.open(BytesIO(img_data))
    tensor = ToTensor()(image).float()
    
    with torch.no_grad():  # Disable gradient calculation for inference
        prediction = model(tensor.unsqueeze(0).to(device))
        _, y_pred = torch.max(prediction.data, 1)
        
    return float(y_pred)
    

    
    

predict_udf = udf(predict_tensor, FloatType())

In [11]:
spark.udf.register("predict_udf", predict_udf)

VECTORIZED UDF (WORKS IN SPARK SQL)

In [12]:
@pandas_udf(returnType=FloatType())
def v_predict_tensor(base64image):
    base64image_decode = base64image.apply(lambda x: base64.b64decode(x))
    image_pil = base64image_decode.apply(lambda x: Image.open(BytesIO(x)))
    tensor_series = image_pil.apply(lambda x: ToTensor()(x).float())
    input_tensor = tensor_series.tolist()
    
    with torch.no_grad():  # Disable gradient calculation for inference
        predictions = model(torch.stack(input_tensor).to(device))
        _, y_pred = torch.max(predictions.data, 1)
    #return y_pred
    return pd.Series(y_pred.squeeze().tolist())


In [13]:
spark.udf.register("v_predict_udf", v_predict_tensor)

In [14]:
#tuning knob 2. put a .limit(20) at the end of the data set loads to reduce memory footprint for dev stuff 
#all_images_labeled = spark.read.parquet("./silver/all_images_labeled.parquet")
labeled_training_set = spark.read.parquet("./silver/labeled_training_set_no_tensor.parquet")
labeled_training_set.cache()
labeled_training_set.count()

998

In [15]:
labeled_training_set.show()

+--------------------+--------------------+----------------+--------------------+----------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+
|                name|           imageData|prelabeled_fraud|                  id|     person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|
+--------------------+--------------------+----------------+--------------------+----------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+
|generated.photos_...|iVBORw0KGgoAAAANS...|               0|generated.photos_...|  JASMINE BRYANT|588 Camelback Roa...|1987-04-30|     F|    black|    A|2022-01-18| 2027-01-18|      BRO|       BRO|   FALSE|     FALSE|     D118

In [16]:
labeled_training_set.select('name', 'id').show(n=1000, truncate=False)

+-------------------------------------------+-------------------------------------------+
|name                                       |id                                         |
+-------------------------------------------+-------------------------------------------+
|generated.photos_v3_0085040_0975120_0785655|generated.photos_v3_0085040_0975120_0785655|
|generated.photos_v3_0093154                |generated.photos_v3_0093154                |
|generated.photos_v3_0045204                |generated.photos_v3_0045204                |
|generated.photos_v3_0060205                |generated.photos_v3_0060205                |
|generated.photos_v3_0007944                |generated.photos_v3_0007944                |
|generated.photos_v3_0034804                |generated.photos_v3_0034804                |
|generated.photos_v3_0091104_0611376_0647554|generated.photos_v3_0091104_0611376_0647554|
|generated.photos_v3_0092266                |generated.photos_v3_0092266                |
|generated

examples of dataframe api udf calls. Faster than sql, but will need to do more dataframe manipulation thats not as easy as spark.sql

In [17]:
test = labeled_training_set.withColumn("mylabel", predict_udf(labeled_training_set['imageData']))
test.show()

+--------------------+--------------------+----------------+--------------------+----------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+-------+
|                name|           imageData|prelabeled_fraud|                  id|     person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|mylabel|
+--------------------+--------------------+----------------+--------------------+----------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+-------+
|generated.photos_...|iVBORw0KGgoAAAANS...|               0|generated.photos_...|  JASMINE BRYANT|588 Camelback Roa...|1987-04-30|     F|    black|    A|2022-01-18| 2027-01-18|      BRO|       BRO|   FA

In [18]:
test2 = labeled_training_set.withColumn("mylabel", v_predict_tensor(labeled_training_set['imageData']))
test2.show()

+--------------------+--------------------+----------------+--------------------+----------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+-------+
|                name|           imageData|prelabeled_fraud|                  id|     person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|mylabel|
+--------------------+--------------------+----------------+--------------------+----------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+-------+
|generated.photos_...|iVBORw0KGgoAAAANS...|               0|generated.photos_...|  JASMINE BRYANT|588 Camelback Roa...|1987-04-30|     F|    black|    A|2022-01-18| 2027-01-18|      BRO|       BRO|   FA

In [19]:
#last tuning know, change reparition based on your system specs and the size of the data being read in. spark will usually do around 20 by default. may or may not work well for you. higher will protect against OOM errors. Lower will make your processing time alot higher
labeled_training_set = labeled_training_set.repartition(threads*10)
labeled_training_set.cache()
labeled_training_set.count()

998

In [20]:
#labeled_training_set.rdd.getNumPartitions()

examples of using udf in sql queries

In [21]:
spark.sql("select gender, predict_udf(imageData) as my_predictions from {labeled_training_set}", labeled_training_set=labeled_training_set).show()

+------+--------------+
|gender|my_predictions|
+------+--------------+
|     M|           0.0|
|     M|           0.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           0.0|
|     M|           0.0|
|     M|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
+------+--------------+
only showing top 20 rows


In [22]:
spark.sql("select gender, v_predict_udf(imageData) as my_predictions from {labeled_training_set}", labeled_training_set=labeled_training_set).show()

+------+--------------+
|gender|my_predictions|
+------+--------------+
|     M|           0.0|
|     M|           0.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           0.0|
|     M|           0.0|
|     M|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
+------+--------------+
only showing top 20 rows


In [23]:
spark.sql("with cte as (select gender, predict_udf(imageData) as my_predictions from {labeled_training_set}) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions", labeled_training_set=labeled_training_set).show()

+------+--------------+-----------------+
|gender|my_predictions|count_predictions|
+------+--------------+-----------------+
|     F|           1.0|              104|
|     F|           0.0|              442|
|     M|           1.0|               92|
|     M|           0.0|              357|
| ERROR|           1.0|                3|
+------+--------------+-----------------+



In [24]:
spark.sql("with cte as (select gender, v_predict_udf(imageData) as my_predictions from {labeled_training_set}) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions", labeled_training_set=labeled_training_set).show()


+------+--------------+-----------------+
|gender|my_predictions|count_predictions|
+------+--------------+-----------------+
|     F|           1.0|              104|
|     F|           0.0|              442|
|     M|           1.0|               92|
|     M|           0.0|              357|
| ERROR|           1.0|                3|
+------+--------------+-----------------+



In [25]:
def pdf_udf(gender, weight_lbs):
    
    #def normpdf(x, mean, sd):
    #    var = float(sd)**2
    #    denom = (2*math.pi*var)**.5
    #    num = math.exp(-(float(x)-float(mean))**2/(2*var))
    #    return num/denom
    
    #randomly taken from google, do not claim this to be sci accurate on stats
    male_mean = 200
    male_dev = 40
    female_mean = 170
    female_dev = 51

    gender = gender.upper()
    
    if gender == 'M':
        a = NormalDist(male_mean, male_dev).cdf(int(weight_lbs)+5)
        b = NormalDist(male_mean, male_dev).cdf(int(weight_lbs)-5)
        probability_density = abs(a-b)
        
    else: 
        a = NormalDist(female_mean, female_dev).cdf(int(weight_lbs)+5)
        b = NormalDist(female_mean, female_dev).cdf(int(weight_lbs)-5)
        probability_density = abs(a-b)
    
    return round(probability_density, 3)

In [26]:
spark.udf.register("pdf_udf_sql", pdf_udf)

<function __main__.pdf_udf(gender, weight_lbs)>

In [27]:
spark.sql('select gender, weight_lbs, pdf_udf_sql(gender, weight_lbs) from {labeled_training_set}', labeled_training_set=labeled_training_set).show()

+------+----------+-------------------------------+
|gender|weight_lbs|pdf_udf_sql(gender, weight_lbs)|
+------+----------+-------------------------------+
|     M|       249|                          0.047|
|     F|       150|                          0.072|
|     M|       117|                          0.012|
|     F|       193|                          0.071|
|     F|       122|                           0.05|
|     F|       139|                          0.065|
|     F|       128|                          0.056|
|     F|       198|                          0.067|
|     M|       175|                          0.082|
|     F|       100|                          0.031|
|     M|       155|                          0.053|
|     M|       117|                          0.012|
|     M|       106|                          0.006|
|     M|       133|                          0.025|
|     M|       248|                          0.049|
|     F|       187|                          0.074|
|     M|    

In [28]:
spark.sql("with cte1 as (select gender, weight_lbs, imageData, pdf_udf_sql(gender, weight_lbs) as w_prob from {labeled_training_set}), cte2 as (select *, v_predict_udf(imageData) as fraud from cte1) select gender, fraud, avg(w_prob), max(w_prob), min(w_prob) from cte2 group by gender, fraud" , labeled_training_set=labeled_training_set).show()

+------+-----+-------------------+-----------+-----------+
|gender|fraud|        avg(w_prob)|max(w_prob)|min(w_prob)|
+------+-----+-------------------+-----------+-----------+
| ERROR|  1.0|0.06133333333333333|      0.077|      0.034|
|     F|  0.0|0.06374886877828055|      0.078|      0.031|
|     F|  1.0|           0.063875|      0.078|        0.0|
|     M|  0.0|0.05880672268907562|      0.099|      0.004|
|     M|  1.0|0.06670652173913041|      0.099|        0.0|
+------+-----+-------------------+-----------+-----------+



In [29]:
spark.sql("with cte as (select gender, weight_lbs, predict_udf(imageData) as my_predictions from {labeled_training_set}) select gender, my_predictions, max(weight_lbs), min(weight_lbs) as count_predictions from cte group by gender, my_predictions", labeled_training_set=labeled_training_set).show()

+------+--------------+---------------+-----------------+
|gender|my_predictions|max(weight_lbs)|count_predictions|
+------+--------------+---------------+-----------------+
| ERROR|           1.0|            178|              104|
|     F|           0.0|            200|              100|
|     F|           1.0|            226|                0|
|     M|           0.0|            249|              100|
|     M|           1.0|            248|                0|
+------+--------------+---------------+-----------------+



In [30]:
log=[]

def monitor_system_resources(interval=1):
    global monitoring_active
    monitoring_active = True
    print("Monitoring system resources...")

    while monitoring_active:
        now = datetime.datetime.now()
        cpu_percent = psutil.cpu_percent(interval=None) # Non-blocking call
        virtual_memory = psutil.virtual_memory()
        memory_percent = virtual_memory.percent
        #print(f"System CPU: {cpu_percent}% | System Memory: {memory_percent}%")
        time.sleep(interval)


        pid_list = []

        for proc in psutil.process_iter(['pid', 'name', 'cpu_percent', 'memory_percent', 'memory_info']):
            try:
                # Get process information
                pid = proc.info['pid']
                name = proc.info['name']
                cpu_percent_pid = proc.cpu_percent(interval=None) # Calculate CPU usage over a short interval
                memory_percent_pid = proc.info['memory_percent']
                if cpu_percent_pid > 1 or  memory_percent_pid > 1:
                    #memory_percent_pid = proc.info['memory_percent']
                    memory_info = proc.info['memory_info']
                    rss_mb = memory_info.rss / (1024 * 1024) if memory_info else 0 # Resident Set Size in MB

                    #print(f"{pid:<8} | {name:<30} | {cpu_percent_pid:<10.2f} | {memory_percent_pid:<10.2f} | {rss_mb:<15.2f}")

                    pid_entry = { "pid": pid,
                                  "name": name,
                                  "cpu_percent": cpu_percent_pid,
                                  "memory_percent": memory_percent_pid,
                                  "rss_mb": rss_mb
                                  }

                    pid_list.append(pid_entry)


            except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
                # Handle cases where a process might disappear or access is denied
                continue

        system_entry = {
            "time" : now.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3] ,
            "cpu_percent" : cpu_percent,
            "memory_percent" : memory_percent,
            #"virtual_memory" : virtual_memory,
            "pids": pid_list

        }

        log.append(system_entry)

    print("Monitoring stopped.")


In [31]:
def my_function_to_trace():
    print("Starting function to trace...")
    spark.sql("with cte as (select gender, v_predict_udf(imageData) as my_predictions from {labeled_training_set}) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions", labeled_training_set=labeled_training_set).show()
    print("Function to trace finished.")

In [32]:
# Start the monitoring thread
monitor_thread = threading.Thread(target=monitor_system_resources, args=(1,))
monitor_thread.start()

# Run the function to be traced
my_function_to_trace()

# Stop the monitoring thread
global monitoring_active
monitoring_active = False
monitor_thread.join() # Wait for the monitoring thread to finish
print("Program finished.")

Monitoring system resources...
Starting function to trace...


+------+--------------+-----------------+
|gender|my_predictions|count_predictions|
+------+--------------+-----------------+
|     F|           1.0|              104|
|     F|           0.0|              442|
|     M|           1.0|               92|
|     M|           0.0|              357|
| ERROR|           1.0|                3|
+------+--------------+-----------------+

Function to trace finished.
Monitoring stopped.
Program finished.


In [33]:
pretty_json = json.dumps(log, indent=4)
print(pretty_json)

[
    {
        "time": "2025-10-05 23:46:48.358",
        "cpu_percent": 0.0,
        "memory_percent": 41.7,
        "pids": [
            {
                "pid": 1033,
                "name": "clickhouse-server",
                "cpu_percent": 0.0,
                "memory_percent": 2.0883837761942594,
                "rss_mb": 1023.56640625
            },
            {
                "pid": 1285,
                "name": "sqlservr",
                "cpu_percent": 0.0,
                "memory_percent": 2.2339863652089753,
                "rss_mb": 1094.9296875
            },
            {
                "pid": 2389,
                "name": "gnome-shell",
                "cpu_percent": 0.0,
                "memory_percent": 1.311786158580658,
                "rss_mb": 642.9375
            },
            {
                "pid": 3339,
                "name": "datagrip",
                "cpu_percent": 0.0,
                "memory_percent": 3.5819799003233164,
                "rss_mb":

In [34]:
xgboostmodel = xgb.XGBClassifier(enable_categorical=True)

In [35]:
xgboostmodel.load_model('xgboost_model.model')

In [36]:
def predict_xgboost(*features):

    feature_list = list(features)
    data = [
        {
            'gender': features[0],
            'ethnicity': features[1],
            'class': features[2],
            'eye_color': features[3],
            'hair_color': features[4],
            'is_donor': features[5],
            'is_veteran': features[6],
            'height_inches': features[7],
            'weight_lbs': features[8]
        }
    ]
    df = pd.DataFrame(data)

    df['gender'] = df['gender'].astype('category')
    df['ethnicity'] = df['ethnicity'].astype('category')
    df['class'] = df['class'].astype('category')
    df['eye_color'] = df['eye_color'].astype('category')
    df['hair_color'] = df['hair_color'].astype('category')
    df['is_donor'] = df['is_donor'].astype('category')
    df['is_veteran'] = df['is_veteran'].astype('category')
    df['height_inches'] = df['height_inches'].astype('int64')
    df['weight_lbs'] = df['weight_lbs'].astype('int64')

    #feature_np = np.array(feature_list, dtype=object)

    y_pred = xgboostmodel.predict(df)

    return float(y_pred)





predict_xgboost_udf = udf(predict_xgboost, FloatType())
spark.udf.register("predict_xgboost_udf", predict_xgboost_udf)

In [37]:
test = spark.sql("select name, prelabeled_fraud, predict_xgboost_udf(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) as my_predictions from {labeled_training_set}", labeled_training_set=labeled_training_set)

In [38]:
test.toPandas()

/tmp/ipykernel_126608/2970470935.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_126608/2970470935.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_126608/2970470935.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_126608/2970470935.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performi

,name,prelabeled_fraud,my_predictions
0,generated.photos_v3_0098917,0,0.0
1,generated.photos_v3_0030752,0,0.0
2,generated.photos_v3_0093154,0,0.0
3,generated.photos_v3_0058749_0041322_0519554,0,0.0
4,generated.photos_v3_0079389_0652990_0192846,0,0.0
...,...,...,...
993,generated.photos_v3_0011839,0,0.0
994,generated.photos_v3_0012976,1,0.0
995,generated.photos_v3_0044557,0,0.0
996,generated.photos_v3_0014397,1,0.0


In [39]:
def predict_xgboost_vector(gender, ethnicity, driver_class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs):


    data = [
        {
            'gender': gender,
            'ethnicity': ethnicity,
            'class': driver_class,
            'eye_color': eye_color,
            'hair_color': hair_color,
            'is_donor': is_donor,
            'is_veteran': is_veteran,
            'height_inches': height_inches,
            'weight_lbs': weight_lbs
        }
    ]
    df = pd.DataFrame(data)

    df['gender'] = df['gender'].astype('category')
    df['ethnicity'] = df['ethnicity'].astype('category')
    df['class'] = df['class'].astype('category')
    df['eye_color'] = df['eye_color'].astype('category')
    df['hair_color'] = df['hair_color'].astype('category')
    df['is_donor'] = df['is_donor'].astype('category')
    df['is_veteran'] = df['is_veteran'].astype('category')
    df['height_inches'] = df['height_inches'].astype('int64')
    df['weight_lbs'] = df['weight_lbs'].astype('int64')

    #feature_np = np.array(feature_list, dtype=object)

    y_pred = xgboostmodel.predict(df)

    return float(y_pred)





predict_xgboost_vector_udf = udf(predict_xgboost_vector, FloatType())
spark.udf.register("predict_xgboost_vector_udf", predict_xgboost_vector_udf)

In [40]:
spark.sql("select name, prelabeled_fraud, predict_xgboost_vector_udf(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) as my_predictions from {labeled_training_set}", labeled_training_set=labeled_training_set).show()

/tmp/ipykernel_126608/4238544487.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_126608/4238544487.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_126608/4238544487.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_126608/4238544487.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performi

+--------------------+----------------+--------------+
|                name|prelabeled_fraud|my_predictions|
+--------------------+----------------+--------------+
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated.photos_...|               0|           0.0|
|generated

/tmp/ipykernel_126608/4238544487.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_126608/4238544487.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_126608/4238544487.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_126608/4238544487.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performi

In [42]:
df = spark.sql("select name, prelabeled_fraud, predict_xgboost_vector_udf(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) as my_predictions from {labeled_training_set}", labeled_training_set=labeled_training_set).toPandas()
df

/tmp/ipykernel_126608/4238544487.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_126608/4238544487.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_126608/4238544487.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_126608/4238544487.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performi

,name,prelabeled_fraud,my_predictions
0,generated.photos_v3_0098917,0,0.0
1,generated.photos_v3_0030752,0,0.0
2,generated.photos_v3_0093154,0,0.0
3,generated.photos_v3_0058749_0041322_0519554,0,0.0
4,generated.photos_v3_0079389_0652990_0192846,0,0.0
...,...,...,...
993,generated.photos_v3_0011839,0,0.0
994,generated.photos_v3_0012976,1,0.0
995,generated.photos_v3_0044557,0,0.0
996,generated.photos_v3_0014397,1,0.0
